# Élaguer un détecteur d'objets pré-entraîné

SSDlite320 MobileNetV3-Large est un détecteur d'objets fait pour les téléphones : 3,4 millions de paramètres, entraîné par torchvision sur COCO (80 classes d'objets). On va le réduire par élagage, et mesurer ce que l'on y gagne et ce que l'on y perd :

1. Mesurer la qualité : la mAP
2. Mesurer le coût : paramètres, taille, latence
3. Élagage non structuré
4. Élagage structuré avec Torch-Pruning
5. Réentraîner le modèle élagué
6. La perte, classe par classe

Ce notebook est prévu pour un GPU : *Exécution > Modifier le type d'exécution > GPU T4*. Sans GPU, il tourne quand même, en plus petit : 60 images au lieu de 943, et une seule époque de réentraînement, dont on ne peut rien conclure.

## Mise en place

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh coco-val2017-open
%pip install -q "torchmetrics[detection]" torch-pruning

In [ ]:
import copy
import gzip
import io
import json
import statistics
import time
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch_pruning as tp
from torch import nn
from torch.nn.utils import prune
from torch.utils.data import DataLoader, Dataset
from torchmetrics.detection import MeanAveragePrecision
from torchvision import tv_tensors
from torchvision.io import ImageReadMode, decode_image
from torchvision.models.detection import (
  SSDLite320_MobileNet_V3_Large_Weights,
  ssdlite320_mobilenet_v3_large,
)
from torchvision.transforms import v2
from torchvision.utils import draw_bounding_boxes

GPU = torch.cuda.is_available()
DEVICE = "cuda" if GPU else "cpu"
# Sans GPU : 60 images au lieu de 943, et une seule époque de réentraînement
N_IMAGES = 943 if GPU else 60
EPOCHS = 10 if GPU else 1
LEARNING_RATE = 3e-4
torch.manual_seed(0)
# Un avertissement de Torch-Pruning sans conséquence sur ce modèle
warnings.filterwarnings("ignore", "Maximum recursive depth", module="torch_pruning")
print(
  "GPU"
  if GPU
  else "Pas de GPU : 60 images et une époque, de quoi vérifier le code seulement"
)

Les données (le dossier `coco-val2017-open` de shuuchuu/datasets, 156 Mo) : 943 images de COCO val2017 (Lin et al., 2014), les 1 000 premières par identifiant dont la photo est sous licence CC BY 2.0 ou CC BY-SA 2.0 et qui contiennent au moins un objet, moins les 57 dont l'auteur a restreint la licence depuis, avec leurs annotations (sous licence CC BY 4.0, comme toutes celles de COCO). L'auteur, la licence et la page Flickr de chaque photo sont dans `coco-val2017-open/credits.csv`. Le modèle n'a jamais vu ces images, torchvision l'a entraîné sur COCO train2017 : la première moitié sert à le réentraîner, la seconde à le tester.

In [ ]:
DATA = Path("coco-val2017-open")
coco = json.loads((DATA / "annotations.json").read_text())
annotations = defaultdict(list)
for annotation in coco["annotations"]:
  annotations[annotation["image_id"]].append(annotation)
infos = sorted(coco["images"], key=lambda info: info["id"])[:N_IMAGES]
images_dir = DATA / "images"
print(
  len(infos), "images,", sum(len(annotations[i["id"]]) for i in infos), "objets annotés"
)

Le modèle redimensionne toute image en 320 × 320 avant de la traiter : on le fait ici une fois pour toutes, boîtes comprises, ce qui garde les images en mémoire (300 Mo) et accélère l'entraînement. Étirer une image ne change pas l'IoU de deux boîtes, donc ne change pas la mAP.

Chaque élément est une image et sa cible : les boîtes `boxes` en (x1, y1, x2, y2), les classes `labels` (les identifiants de COCO, de 1 à 90) et `iscrowd` (1 pour une boîte qui entoure une foule, que la mAP de COCO ne compte ni comme trouvée ni comme manquée). L'entraînement n'utilise pas ces boîtes de foule, et retourne une image sur deux horizontalement.

In [ ]:
SIZE = 320


def load(info: dict, crowds: bool) -> tuple[tv_tensors.Image, dict]:
  image = decode_image(str(images_dir / info["file_name"]), mode=ImageReadMode.RGB)
  sx, sy = SIZE / info["width"], SIZE / info["height"]
  kept = [
    a
    for a in annotations[info["id"]]
    if min(a["bbox"][2:]) >= 1 and (crowds or not a["iscrowd"])
  ]
  # Les boîtes de COCO sont en (x, y, largeur, hauteur), en pixels
  boxes = [
    [x * sx, y * sy, (x + w) * sx, (y + h) * sy]
    for x, y, w, h in (a["bbox"] for a in kept)
  ]
  target = {
    "boxes": tv_tensors.BoundingBoxes(
      torch.tensor(boxes).reshape(-1, 4), format="XYXY", canvas_size=(SIZE, SIZE)
    ),
    "labels": torch.tensor([a["category_id"] for a in kept], dtype=torch.int64),
    "iscrowd": torch.tensor([a["iscrowd"] for a in kept], dtype=torch.int64),
  }
  return tv_tensors.Image(v2.functional.resize(image, [SIZE, SIZE])), target


class Detection(Dataset):
  def __init__(self, items: list, train: bool = False) -> None:
    self.items = items
    flip = [v2.RandomHorizontalFlip()] if train else []
    self.transform = v2.Compose([*flip, v2.ToDtype(torch.float32, scale=True)])

  def __len__(self) -> int:
    return len(self.items)

  def __getitem__(self, i: int) -> tuple[torch.Tensor, dict]:
    return self.transform(*self.items[i])


def collate(batch: list) -> tuple:
  return tuple(zip(*batch))


half = len(infos) // 2
train_set = Detection([load(info, crowds=False) for info in infos[:half]], train=True)
test_set = Detection([load(info, crowds=True) for info in infos[half:]])
train_loader = DataLoader(train_set, batch_size=16, shuffle=True, collate_fn=collate)
test_loader = DataLoader(test_set, batch_size=16, collate_fn=collate)
print(len(train_set), "images d'entraînement,", len(test_set), "de test")

Le modèle prend une liste d'images (des tenseurs 3 × H × W de flottants entre 0 et 1) et, en mode évaluation, renvoie pour chacune un dictionnaire : `boxes`, `scores` et `labels`, jusqu'à 300 détections triées par score. Voici celles de score supérieur à 0,5 sur une image de test :

In [ ]:
WEIGHTS = SSDLite320_MobileNet_V3_Large_Weights.DEFAULT
categories = WEIGHTS.meta["categories"]  # le nom de chaque identifiant de classe
model = ssdlite320_mobilenet_v3_large(weights=WEIGHTS).to(DEVICE).eval()

image, target = test_set[0]
with torch.inference_mode():
  (prediction,) = model([image.to(DEVICE)])
keep = prediction["scores"] > 0.5
drawn = draw_bounding_boxes(
  v2.functional.to_dtype(image, torch.uint8, scale=True),
  prediction["boxes"][keep].cpu(),
  labels=[categories[i] for i in prediction["labels"][keep].tolist()],
  colors="red",
  width=2,
)
plt.imshow(drawn.permute(1, 2, 0))
plt.axis("off")
plt.show()

## 1. Mesurer la qualité : la mAP

La mAP (*mean average precision*) de COCO : pour chaque classe, l'aire sous la courbe précision-rappel de ses détections, une détection étant juste si son IoU avec une boîte de la même classe dépasse un seuil ; puis la moyenne sur les classes, et sur les seuils d'IoU de 0,5 à 0,95. `map_50` est celle du seul seuil 0,5.

Écrivez `evaluate(model, loader, class_metrics=False)`, qui passe le modèle en mode évaluation, prédit les images de `loader` sous `torch.inference_mode()`, et renvoie le résultat d'une `MeanAveragePrecision` mise à jour lot par lot. Mesurez la mAP du modèle pré-entraîné sur `test_loader`. torchvision annonce 21,3 % (0,213) sur tout COCO val2017 : obtenez-vous la même chose sur ces 500 images ?

**Documentation utile**

- [`MeanAveragePrecision`](https://lightning.ai/docs/torchmetrics/stable/detection/mean_average_precision.html)
- [`ssdlite320_mobilenet_v3_large`](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.detection.ssdlite320_mobilenet_v3_large.html)
- [`torch.inference_mode`](https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad_mode.inference_mode.html)

In [ ]:
# Votre code ici
def evaluate(
  model: nn.Module, loader: DataLoader, class_metrics: bool = False
) -> dict: ...

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `model.eval()`, puis tout le reste sous `with torch.inference_mode():`
- `MeanAveragePrecision(iou_type="bbox", class_metrics=class_metrics)` : les boîtes en (x1, y1, x2, y2), son format par défaut ; `class_metrics=True` ajoute la précision moyenne de chaque classe (`map_per_class`, pour l'exercice 6)
- `metric.warn_on_many_detections = False` : le modèle renvoie jusqu'à 300 détections par image, la mAP de COCO ne compte que les 100 meilleures, et `MeanAveragePrecision` le rappelle à chaque lot
- chaque lot de `loader` est un tuple d'images et un tuple de cibles : `for images, targets in loader:`
- `model([image.to(DEVICE) for image in images])` renvoie la liste des prédictions
- `metric.update(predictions, targets)` : deux listes de dictionnaires, sur le même appareil ; ramenez les prédictions sur le CPU, où sont les cibles (`{k: v.cpu() for k, v in p.items()}`)
- `metric.compute()` renvoie un dictionnaire de tenseurs : `float(results["map"])`, `float(results["map_50"])`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def evaluate(model: nn.Module, loader: DataLoader, class_metrics: bool = False) -> dict:
  model.eval()
  metric = MeanAveragePrecision(iou_type="bbox", class_metrics=...)
  metric.warn_on_many_detections = False
  with torch.inference_mode():
    for images, targets in loader:
      predictions = model(...)
      metric.update([{k: v.cpu() for k, v in p.items()} for p in predictions], ...)
  return metric.compute()


base_metrics = evaluate(model, test_loader)
print(
  f"mAP {float(base_metrics['map']):.3f}, mAP@0.5 {float(base_metrics['map_50']):.3f}"
)
```

### Solution

In [ ]:
def evaluate(model: nn.Module, loader: DataLoader, class_metrics: bool = False) -> dict:
  model.eval()
  metric = MeanAveragePrecision(iou_type="bbox", class_metrics=class_metrics)
  # La mAP de COCO ne compte que les 100 meilleures détections par image
  metric.warn_on_many_detections = False
  with torch.inference_mode():
    for images, targets in loader:
      predictions = model([image.to(DEVICE) for image in images])
      metric.update([{k: v.cpu() for k, v in p.items()} for p in predictions], targets)
  return metric.compute()


base_metrics = evaluate(model, test_loader)
print(
  f"mAP {float(base_metrics['map']):.3f}, mAP@0.5 {float(base_metrics['map_50']):.3f}"
)

## 2. Mesurer le coût : paramètres, taille, latence

Écrivez `profile(model)`, qui renvoie un dictionnaire :

- `params_M` : le nombre de paramètres du modèle, en millions ;
- `size_MB` : la taille du fichier de ses poids (`torch.save(model.state_dict(), ...)`), en Mo ;
- `gzip_MB` : la taille de ce fichier compressé par gzip, en Mo ;
- `latency_ms` : la latence médiane d'une prédiction d'une seule image (*batch* de 1), en millisecondes, **sur le CPU** : SSDlite est fait pour les téléphones et les processeurs embarqués, son matériel cible. Prédisez `image` 25 fois avec une copie du modèle sur le CPU, et ignorez les 5 premières mesures (la chauffe).

Une latence se mesure sur une machine au repos : sur Colab, elle varie de quelques pour cent d'une exécution à l'autre.

**Documentation utile**

- [`torch.save`](https://docs.pytorch.org/docs/stable/generated/torch.save.html)
- [`io.BytesIO`](https://docs.python.org/3/library/io.html#io.BytesIO)
- [`gzip.compress`](https://docs.python.org/3/library/gzip.html#gzip.compress)
- [`copy.deepcopy`](https://docs.python.org/3/library/copy.html#copy.deepcopy)
- [`time.perf_counter`](https://docs.python.org/3/library/time.html#time.perf_counter)
- [`statistics.median`](https://docs.python.org/3/library/statistics.html#statistics.median)

In [ ]:
image = test_set[0][0]  # l'image des mesures de latence


# Votre code ici
def profile(model: nn.Module) -> dict: ...

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `sum(p.numel() for p in model.parameters())` : le nombre de paramètres, 3,44 millions pour le modèle pré-entraîné
- `buffer = io.BytesIO()`, puis `torch.save(model.state_dict(), buffer)` : le fichier en mémoire ; `buffer.getvalue()` en donne les octets
- `len(gzip.compress(octets))` : la taille compressée
- `cpu_model = copy.deepcopy(model).cpu().eval()` : une copie, pour ne pas déplacer le modèle lui-même
- `start = time.perf_counter()`, puis `cpu_model([image])`, puis `time.perf_counter() - start`, en secondes, sous `torch.inference_mode()`
- `statistics.median(times[5:])` : la médiane, sans la chauffe

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def profile(model: nn.Module) -> dict:
  buffer = io.BytesIO()
  torch.save(model.state_dict(), buffer)
  cpu_model = copy.deepcopy(model).cpu().eval()
  times = []
  with torch.inference_mode():
    for _ in range(25):
      start = time.perf_counter()
      ...
      times.append(time.perf_counter() - start)
  return {
    "params_M": sum(p.numel() for p in model.parameters()) / 1e6,
    "size_MB": len(buffer.getvalue()) / 1e6,
    "gzip_MB": ...,
    "latency_ms": 1000 * statistics.median(...),
  }


profile(model)
```

### Solution

In [ ]:
def profile(model: nn.Module) -> dict:
  buffer = io.BytesIO()
  torch.save(model.state_dict(), buffer)
  cpu_model = copy.deepcopy(model).cpu().eval()
  times = []
  with torch.inference_mode():
    for _ in range(25):
      start = time.perf_counter()
      cpu_model([image])
      times.append(time.perf_counter() - start)
  return {
    "params_M": sum(p.numel() for p in model.parameters()) / 1e6,
    "size_MB": len(buffer.getvalue()) / 1e6,
    "gzip_MB": len(gzip.compress(buffer.getvalue())) / 1e6,
    "latency_ms": 1000 * statistics.median(times[5:]),
  }


profile(model)

Le tableau des résultats, que chaque exercice complète d'une ligne :

In [ ]:
results = {}


def record(name: str, model: nn.Module, metrics: dict) -> pd.DataFrame:
  """Ajoute la ligne du modèle au tableau, et affiche le tableau."""
  results[name] = {
    "mAP": float(metrics["map"]),
    "mAP@0.5": float(metrics["map_50"]),
    **profile(model),
  }
  return pd.DataFrame(results).T.round(3)


record("pré-entraîné", model, base_metrics)

## 3. Élagage non structuré

Sur une copie du modèle (`unstructured`), mettez à zéro la moitié des poids des couches `nn.Conv2d` et `nn.Linear`, ceux de plus petite magnitude sur l'ensemble du réseau (`prune.global_unstructured`, comme dans les slides), puis rendez l'élagage définitif (`prune.remove`). Affichez la part de poids nuls, puis la ligne du modèle dans le tableau. Qu'est-ce qui a changé : la mAP, le nombre de paramètres, la taille du fichier, sa taille compressée, la latence ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `unstructured = copy.deepcopy(model)` : le modèle pré-entraîné reste intact
- `[(m, "weight") for m in unstructured.modules() if isinstance(m, nn.Conv2d | nn.Linear)]` : les poids à élaguer
- `prune.global_unstructured(weights, pruning_method=prune.L1Unstructured, amount=0.5)` : le seuil est commun à tout le réseau, donc certaines couches perdent plus de la moitié de leurs poids, d'autres moins
- `prune.remove(module, "weight")` pour chacune : le masque devient définitif
- la part de zéros : le nombre de poids nuls (`(m.weight == 0).sum().item()`) sur le nombre de poids (`m.weight.numel()`), 50 % attendus
- `record("non structuré 50 %", unstructured, evaluate(unstructured, test_loader))`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
unstructured = copy.deepcopy(model)
kinds = nn.Conv2d | nn.Linear
weights = [(m, "weight") for m in unstructured.modules() if isinstance(m, kinds)]
prune.global_unstructured(weights, pruning_method=..., amount=...)
for module, name in weights:
  ...
zeros = sum((m.weight == 0).sum().item() for m, _ in weights)
total = sum(m.weight.numel() for m, _ in weights)
print(f"{zeros / total:.0%}")
record(..., unstructured, evaluate(unstructured, test_loader))
```

### Solution

In [ ]:
unstructured = copy.deepcopy(model)
kinds = nn.Conv2d | nn.Linear
weights = [(m, "weight") for m in unstructured.modules() if isinstance(m, kinds)]
prune.global_unstructured(weights, pruning_method=prune.L1Unstructured, amount=0.5)
for module, name in weights:
  prune.remove(module, name)
zeros = sum((m.weight == 0).sum().item() for m, _ in weights)
total = sum(m.weight.numel() for m, _ in weights)
print(f"{zeros / total:.0%} de poids nuls")
record("non structuré 50 %", unstructured, evaluate(unstructured, test_loader))

## 4. Élagage structuré avec Torch-Pruning

L'élagage structuré retire des canaux entiers : les tenseurs rétrécissent. Sur une autre copie du modèle (`structured`), retirez 20 % des canaux de chaque couche avec un `tp.pruner.MetaPruner` et une `tp.importance.MagnitudeImportance(p=2)`, comme dans les slides (qui en retirent la moitié), en protégeant les têtes de détection (`structured.head`) : leurs sorties, pour chaque ancre les scores des 91 identifiants de classe et les 4 coordonnées d'une boîte, sont fixées par la tâche. Torch-Pruning suit les calculs du modèle sur un exemple d'entrée : ici, une liste d'une image de 3 × 320 × 320 sur `DEVICE`. Affichez la ligne du modèle dans le tableau : que valent la mAP, la taille et la latence ?

**Documentation utile**

- [Torch-Pruning](https://github.com/VainF/Torch-Pruning)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `structured = copy.deepcopy(model)`
- `example = [torch.rand(3, 320, 320, device=DEVICE)]` : le modèle prend une liste d'images
- `tp.pruner.MetaPruner(structured, example, importance=tp.importance.MagnitudeImportance(p=2), pruning_ratio=0.2, ignored_layers=[structured.head])` : les canaux de plus petite norme L2 partent, 20 % de ceux de chaque couche
- `pruner.step()` les retire, une fois pour toutes ; dans les têtes, Torch-Pruning retire aussi les canaux d'entrée qui correspondaient aux canaux retirés : seules leurs sorties sont protégées
- `record("structuré 20 %", structured, evaluate(structured, test_loader))`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
structured = copy.deepcopy(model)
example = [torch.rand(3, 320, 320, device=DEVICE)]
pruner = tp.pruner.MetaPruner(
  structured,
  example,
  importance=...,
  pruning_ratio=...,
  ignored_layers=[...],
)
pruner.step()
record(..., structured, evaluate(structured, test_loader))
```

### Solution

In [ ]:
structured = copy.deepcopy(model)
example = [torch.rand(3, 320, 320, device=DEVICE)]
pruner = tp.pruner.MetaPruner(
  structured,
  example,
  importance=tp.importance.MagnitudeImportance(p=2),
  pruning_ratio=0.2,
  ignored_layers=[structured.head],
)
pruner.step()
record("structuré 20 %", structured, evaluate(structured, test_loader))

## 5. Réentraîner le modèle élagué

Écrivez `train_epoch(model, loader, optimizer)` : une époque d'entraînement, qui renvoie la perte moyenne. En mode entraînement (`model.train()`), le détecteur prend les images **et** leurs cibles, et renvoie ses pertes, un dictionnaire : `bbox_regression` (les boîtes) et `classification` (les classes) ; la perte à minimiser est leur somme. Réentraînez `structured` pendant `EPOCHS` époques sur `train_loader`, avec `torch.optim.AdamW` et un taux d'apprentissage de `LEARNING_RATE`, en affichant la perte de chaque époque, puis affichez la ligne du modèle dans le tableau. Quelle part de la mAP perdue regagne-t-on ?

**Documentation utile**

- [`torch.optim.AdamW`](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html)

In [ ]:
# Votre code ici
def train_epoch(
  model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer
) -> float: ...

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `model.train()` au début de l'époque (`evaluate` repasse le modèle en mode évaluation)
- les images et les cibles sur `DEVICE` : `[image.to(DEVICE) for image in images]`, `[{k: v.to(DEVICE) for k, v in t.items()} for t in targets]`
- `losses = model(images, targets)`, puis `loss = sum(losses.values())`
- `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()` ; `loss.item()` pour la moyenne
- `torch.optim.AdamW(structured.parameters(), lr=LEARNING_RATE)`, créé une fois, avant la boucle des époques
- `record("structuré 20 %, réentraîné", structured, evaluate(structured, test_loader))`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def train_epoch(
  model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer
) -> float:
  model.train()
  total = 0.0
  for images, targets in loader:
    images = [image.to(DEVICE) for image in images]
    targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
    losses = model(...)
    loss = ...
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total += loss.item()
  return total / len(loader)


optimizer = torch.optim.AdamW(structured.parameters(), lr=LEARNING_RATE)
for epoch in range(EPOCHS):
  print(epoch + 1, train_epoch(structured, train_loader, optimizer))
record(..., structured, evaluate(structured, test_loader))
```

### Solution

In [ ]:
def train_epoch(
  model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer
) -> float:
  model.train()
  total = 0.0
  for images, targets in loader:
    images = [image.to(DEVICE) for image in images]
    targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
    losses = model(images, targets)
    loss = sum(losses.values())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total += loss.item()
  return total / len(loader)


optimizer = torch.optim.AdamW(structured.parameters(), lr=LEARNING_RATE)
for epoch in range(EPOCHS):
  loss = train_epoch(structured, train_loader, optimizer)
  print(f"époque {epoch + 1} : perte {loss:.3f}")
record("structuré 20 %, réentraîné", structured, evaluate(structured, test_loader))

## 6. La perte, classe par classe

Une mAP globale peut cacher des pertes concentrées sur quelques classes : un réseau élagué oublie d'abord les exemples rares et difficiles (Hooker et al., 2019, *What Do Compressed Deep Neural Networks Forget?*). Évaluez le modèle pré-entraîné et le modèle réentraîné avec `class_metrics=True`, puis construisez un `DataFrame` indexé par le nom de la classe, avec le nombre d'objets de la classe dans les images de test, la précision moyenne (AP) avant et après l'élagage, et leur écart. Gardez les classes d'au moins `MIN_OBJECTS` objets, et triez-les par écart. Toutes les classes perdent-elles autant ? Celles qui perdent le plus sont-elles les plus rares ?

**Documentation utile**

- [`collections.Counter`](https://docs.python.org/3/library/collections.html#collections.Counter)
- [`pandas.DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [`DataFrame.sort_values`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html)

In [ ]:
# Moins d'objets : l'AP de la classe dépend trop des quelques images tirées
MIN_OBJECTS = 20 if GPU else 1

# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `evaluate(model, test_loader, class_metrics=True)` et de même pour `structured` : `metrics["map_per_class"]` donne l'AP de chaque classe, dans l'ordre des identifiants de `metrics["classes"]` ; `pd.Series(aps, index=identifiants)` les associe
- les objets de chaque classe : un `Counter` des `labels` des cibles de `test_set.items`, sans les boîtes de foule (`iscrowd` à 1)
- `pd.DataFrame({"objets": ..., "AP avant": ..., "AP après": ...})` aligne les trois séries sur leurs identifiants ; `classes.index = [categories[i] for i in classes.index]` les remplace par les noms
- `classes[classes["objets"] >= MIN_OBJECTS].sort_values("écart")` : la classe qui perd le plus en premier

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def per_class(metrics: dict) -> pd.Series:
  return pd.Series(metrics["map_per_class"].tolist(), index=metrics["classes"].tolist())


before = per_class(evaluate(model, test_loader, class_metrics=True))
after = per_class(...)
objects = Counter(
  label
  for _, target in test_set.items
  for label, crowd in zip(target["labels"].tolist(), target["iscrowd"].tolist())
  if not crowd
)
classes = pd.DataFrame(
  {"objets": pd.Series(objects), "AP avant": before, "AP après": after}
)
classes["écart"] = ...
classes.index = [categories[i] for i in classes.index]
classes[classes["objets"] >= MIN_OBJECTS].sort_values(...).round(3)
```

### Solution

In [ ]:
def per_class(metrics: dict) -> pd.Series:
  return pd.Series(metrics["map_per_class"].tolist(), index=metrics["classes"].tolist())


before = per_class(evaluate(model, test_loader, class_metrics=True))
after = per_class(evaluate(structured, test_loader, class_metrics=True))
objects = Counter(
  label
  for _, target in test_set.items
  for label, crowd in zip(target["labels"].tolist(), target["iscrowd"].tolist())
  if not crowd
)
classes = pd.DataFrame(
  {"objets": pd.Series(objects), "AP avant": before, "AP après": after}
)
classes["écart"] = classes["AP après"] - classes["AP avant"]
classes.index = [categories[i] for i in classes.index]
classes[classes["objets"] >= MIN_OBJECTS].sort_values("écart").round(3)

## Bilan

Les quatre modèles côte à côte :

In [ ]:
pd.DataFrame(results).T.round(3)

- Lequel de ces modèles déploieriez-vous sur un téléphone ? Que faudrait-il encore mesurer, et où (sur le téléphone lui-même, avec son moteur d'inférence) ?
- Retirer la moitié des canaux, comme dans les slides, ramène la mAP de ce détecteur à presque 0, et quelques époques sur 500 images n'en regagnent presque rien : un réseau déjà compact comme MobileNetV3 a peu de canaux inutiles.
- Pour regagner plus de mAP : réentraîner plus longtemps, sur COCO train2017 (118 000 images) ; élaguer progressivement, par petites étapes séparées de réentraînements (un plan d'élagage) ; retirer moins de canaux ; distiller le modèle d'origine dans le modèle élagué.
- L'élagage non structuré n'a réduit que le fichier compressé : à quoi peut-il servir quand même ?